# Movie Review Sentiment Classification Using Machine Learning

**Course:** DSA 495

**Team Members:** Prisha Shah, Laura Botnaru, Grishma Kuikel

## 1. Project Introduction

### Objective
The objective of this project is to classify movie reviews as positive or negative using natural language processing (NLP) and machine learning.

### Research Question
How effectively can machine learning classify the sentiment of written movie reviews using TF-IDF and Logistic Regression?

### Dataset
We use the IMDb Large Movie Review Dataset, which contains 50,000 labeled movie reviews divided into training and testing datasets.

### Approach
- Collect and explore the IMDb dataset.
- Clean and prepare the review text.
- Convert text into numerical features using TF-IDF.
- Train a baseline classifier and a Logistic Regression model.
- Evaluate model performance using accuracy, precision, recall, and F1-score.
- Analyze misclassified reviews and identify model limitations.

### Success Criteria
The Logistic Regression model should outperform a majority-class baseline on unseen movie reviews.

## 2. Environment Setup and Imports

This section installs and imports the Python libraries required for data collection, text preprocessing, machine learning, evaluation, and visualization.

The analysis is designed to run in Google Colab using a standard CPU runtime. A fixed random seed is used to support reproducibility.

In [ ]:
# Install the Hugging Face datasets library
%pip install -q datasets

from datasets import load_dataset
import pandas as pd
import numpy as np

In [ ]:
import sys
import importlib.metadata
from pathlib import Path

# Record Python and package versions
print("Python version:", sys.version.split()[0])

packages = [
    "datasets",
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "seaborn"
]

requirements = []

for package in packages:
    version = importlib.metadata.version(package)
    print(f"{package}: {version}")
    requirements.append(f"{package}=={version}")

# Save package versions for reproducibility
Path("requirements.txt").write_text(
    "\n".join(requirements) + "\n"
)

print("\nrequirements.txt created successfully!")

In [ ]:
# Text preprocessing
import re
import html

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File management and reproducibility
from pathlib import Path
import sys
import importlib.metadata

# Set random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Create output folders
Path("figures").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

print("All libraries imported successfully!")

## 3. Data Collection

### Dataset Source

This project uses the IMDb Large Movie Review Dataset introduced by Maas et al. (2011) and accessed through Hugging Face.

- **Source:** https://huggingface.co/datasets/stanfordnlp/imdb
- **Original documentation:** https://ai.stanford.edu/~amaas/data/sentiment/
- **Unit of analysis:** One movie review
- **Input variable:** Review text
- **Target variable:** Sentiment label
- **Label 0:** Negative sentiment
- **Label 1:** Positive sentiment

The dataset contains 25,000 labeled training reviews, 25,000 labeled testing reviews, and 50,000 unlabeled reviews. We use only the labeled reviews because this project focuses on supervised binary classification.

We preserve the original testing split for final evaluation and create a validation split from the original training data.

The raw dataset will be retrieved programmatically rather than redistributed through GitHub.

In [ ]:
# Load the IMDb dataset from Hugging Face
dataset = load_dataset("stanfordnlp/imdb")

# Inspect the dataset structure
print(dataset)

In [ ]:
# Convert labeled datasets into Pandas DataFrames
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

# Display dataset dimensions
print("Training dataset shape:", train_df.shape)
print("Testing dataset shape:", test_df.shape)

# Display column names
print("\nColumn names:", train_df.columns.tolist())

# Preview the first five training reviews
train_df.head()

## 4. Exploratory Data Analysis (EDA)

Before training the model, we examine the dataset to understand its characteristics and identify potential data quality issues.

This section examines:

- Missing and empty reviews
- Duplicate reviews
- Overlapping training and testing reviews
- Sentiment class distribution
- Review length distribution

These checks help identify potential class imbalance, data leakage, and data quality issues that may affect model performance.

In [ ]:
# Check data quality for each dataset
def check_data_quality(df, name):
    print(f"\n{name} Dataset")
    print("-" * 40)
    print("Total reviews:", len(df))
    print("Missing text:", df["text"].isna().sum())
    print("Missing labels:", df["label"].isna().sum())
    print("Empty reviews:",
          df["text"].fillna("").str.strip().eq("").sum())
    print("Duplicate reviews:",
          df["text"].duplicated().sum())
    print("Unique labels:", sorted(df["label"].unique()))

# Run quality checks
check_data_quality(train_df, "Training")
check_data_quality(test_df, "Testing")

# Check for overlapping reviews between datasets
overlap = set(train_df["text"]) & set(test_df["text"])

print("\nOverlapping training and testing reviews:", len(overlap))

### Data Quality Findings

Both the training and testing datasets contain 25,000 reviews with no missing text, missing labels, or empty reviews.

However, the training dataset contains 96 duplicate reviews, and the testing dataset contains 199 duplicate reviews. Additionally, 123 unique reviews appear in both datasets.

These overlapping reviews create a potential data leakage risk and could inflate model performance. We will remove duplicate and overlapping reviews during text preparation to ensure a more reliable evaluation.

In [ ]:
# Create readable sentiment labels
sentiment_labels = {0: "Negative", 1: "Positive"}

train_df["sentiment"] = train_df["label"].map(sentiment_labels)
test_df["sentiment"] = test_df["label"].map(sentiment_labels)

# Calculate sentiment distribution
sentiment_counts = pd.DataFrame({
    "Training": train_df["sentiment"].value_counts(),
    "Testing": test_df["sentiment"].value_counts()
})

display(sentiment_counts)

# Visualize sentiment distribution
ax = sentiment_counts.plot(
    kind="bar",
    figsize=(8, 5),
    color=["#4472C4", "#ED7D31"],
    rot=0
)

plt.title("Sentiment Distribution by Dataset")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.legend(title="Dataset")

# Add data labels
for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

# Save visualization
plt.savefig(
    "figures/sentiment_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Sentiment Distribution Findings

Both the training and testing datasets contain 12,500 positive reviews and 12,500 negative reviews, resulting in an equal distribution of sentiment classes.

The original dataset is balanced, so class imbalance is not a major concern. A majority-class classifier would achieve 50% accuracy on the original testing dataset, providing a useful baseline for evaluating our Logistic Regression model.

We will check the class distribution again after removing duplicate and overlapping reviews.

In [ ]:
# Calculate review lengths in words
train_df["word_count"] = train_df["text"].str.split().str.len()
test_df["word_count"] = test_df["text"].str.split().str.len()

# Calculate summary statistics for training reviews
length_summary = train_df.groupby("sentiment")["word_count"].agg(
    ["count", "mean", "median", "min", "max", "std"]
).round(2)

display(length_summary)

# Visualize review length distribution
plt.figure(figsize=(10, 5))

sns.histplot(
    data=train_df,
    x="word_count",
    hue="sentiment",
    bins=50,
    element="step",
    common_norm=False,
    palette={
        "Negative": "#D9534F",
        "Positive": "#2E8B57"
    }
)

plt.title("Movie Review Length by Sentiment")
plt.xlabel("Number of Words")
plt.ylabel("Number of Reviews")
plt.tight_layout()

# Save visualization
plt.savefig(
    "figures/review_lengths.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Review Length Findings

Negative reviews contain an average of 230.87 words, while positive reviews contain an average of 236.71 words. Both sentiment classes have a median of 174 words.

The distributions are right-skewed, meaning that most reviews are relatively short, while a smaller number of reviews are substantially longer.

The distributions overlap considerably, suggesting that review length alone is unlikely to be a strong predictor of sentiment. Therefore, our model will focus on the words and phrases within each review using TF-IDF rather than relying on review length alone.

In [ ]:
# Save sentiment distribution results
sentiment_counts.to_csv(
    "results/sentiment_distribution.csv"
)

# Save review length summary statistics
length_summary.to_csv(
    "results/review_length_summary.csv"
)

# Verify saved files
print("Saved figures:")
for file in sorted(Path("figures").glob("*.png")):
    print(file)

print("\nSaved results:")
for file in sorted(Path("results").glob("*.csv")):
    print(file)

print("\nEDA results and visualizations saved successfully!")

## 5. Text Cleaning and Preparation

Before converting the movie reviews into numerical features, we clean and prepare the raw text to improve consistency and reduce potential data leakage.

### Preprocessing Steps

- Decode HTML entities.
- Remove HTML tags and formatting.
- Convert text to lowercase.
- Remove unnecessary whitespace.
- Remove missing or empty reviews.
- Remove duplicate reviews and address conflicting labels.
- Remove overlapping reviews from the training dataset to protect the testing dataset.
- Create a separate validation dataset using a stratified split.

### Preprocessing Decisions

We retain stop words, including words such as "not" and "never," because they can change the meaning and sentiment of a review.

We also avoid stemming and lemmatization to preserve the original wording. Tokenization will be performed during TF-IDF feature extraction.

All learned text representation and modeling steps will be fitted using training data only to prevent data leakage.

In [ ]:
# Define a reusable text-cleaning function
def clean_text(text):
    # Handle missing or invalid text
    if not isinstance(text, str):
        return ""

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Convert text to lowercase
    text = text.lower()

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Test the cleaning function
sample_text = "The movie was NOT good! <br /><br /> Acting &amp; plot were terrible."

print("Original text:")
print(sample_text)

print("\nCleaned text:")
print(clean_text(sample_text))

In [ ]:
# Create copies to preserve the original datasets
train_clean = train_df.copy()
test_clean = test_df.copy()

# Apply the text-cleaning function
train_clean["clean_text"] = train_clean["text"].apply(clean_text)
test_clean["clean_text"] = test_clean["text"].apply(clean_text)

# Check for empty reviews after cleaning
print("Empty training reviews after cleaning:",
      train_clean["clean_text"].eq("").sum())

print("Empty testing reviews after cleaning:",
      test_clean["clean_text"].eq("").sum())

# Compare original and cleaned text
print("\nOriginal review:")
print(train_clean["text"].iloc[0][:500])

print("\nCleaned review:")
print(train_clean["clean_text"].iloc[0][:500])

# Verify dataset dimensions
print("\nTraining dataset shape:", train_clean.shape)
print("Testing dataset shape:", test_clean.shape)

In [ ]:
# Define a reusable function to remove invalid and duplicate reviews
def remove_duplicates(df, name):
    original_count = len(df)

    # Remove missing labels and empty reviews
    df = df.dropna(subset=["label"]).copy()
    df = df[df["clean_text"] != ""].copy()

    # Identify identical reviews with conflicting sentiment labels
    label_counts = df.groupby("clean_text")["label"].nunique()
    conflicting_reviews = label_counts[label_counts > 1].index

    # Remove reviews with conflicting labels
    df = df[~df["clean_text"].isin(conflicting_reviews)]

    # Remove duplicate cleaned reviews
    df = df.drop_duplicates(subset=["clean_text"])

    print(f"{name} Dataset")
    print("Original reviews:", original_count)
    print("Unique texts with conflicting labels:", len(conflicting_reviews))
    print("Total reviews removed:", original_count - len(df))
    print("Remaining reviews:", len(df))
    print()

    return df.reset_index(drop=True)

# Remove duplicates within each dataset
train_clean = remove_duplicates(train_clean, "Training")
test_clean = remove_duplicates(test_clean, "Testing")

# Identify overlapping reviews after text cleaning
overlapping_reviews = set(train_clean["clean_text"]) & set(test_clean["clean_text"])

print("Overlapping reviews after cleaning:", len(overlapping_reviews))

# Remove overlapping reviews from training data
train_clean = train_clean[
    ~train_clean["clean_text"].isin(overlapping_reviews)
].reset_index(drop=True)

# Verify that no overlapping reviews remain
remaining_overlap = set(train_clean["clean_text"]) & set(test_clean["clean_text"])

print("Remaining overlapping reviews:", len(remaining_overlap))

# Display final dataset sizes
print("\nFinal training reviews:", len(train_clean))
print("Final testing reviews:", len(test_clean))

# Validate that duplicates and overlap have been removed
assert train_clean["clean_text"].is_unique
assert test_clean["clean_text"].is_unique
assert len(remaining_overlap) == 0

print("\nAll duplicate and overlap checks passed!")

### Data Cleaning Findings

Text preprocessing was successfully applied to all 50,000 labeled movie reviews. No reviews became empty after cleaning, and no identical reviews with conflicting labels were identified within either dataset.

After text normalization, 98 duplicate training reviews and 201 duplicate testing reviews were removed. Additionally, 123 overlapping reviews were removed from the training dataset to prevent exact-text data leakage.

The final cleaned datasets contain:

- Training dataset: 24,779 reviews
- Testing dataset: 24,799 reviews
- Total usable reviews: 49,578
- Total reviews removed: 422
- Percentage of original data retained: 99.16%

All automated duplicate and overlap checks passed. These preprocessing steps improve data quality and support a more reliable evaluation of model performance on unseen reviews.

Although exact duplicate reviews have been eliminated, similar or paraphrased reviews may still exist across datasets.

In [ ]:
# Split the cleaned training dataset into training and validation sets
train_final, val_final = train_test_split(
    train_clean,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=train_clean["label"]
)

# Preserve the cleaned testing dataset
test_final = test_clean.copy()

# Reset dataset indexes
train_final = train_final.reset_index(drop=True)
val_final = val_final.reset_index(drop=True)
test_final = test_final.reset_index(drop=True)

# Display dataset sizes
print("Training reviews:", len(train_final))
print("Validation reviews:", len(val_final))
print("Testing reviews:", len(test_final))

# Compare sentiment distributions
split_summary = pd.DataFrame({
    "Training": train_final["label"].value_counts(),
    "Validation": val_final["label"].value_counts(),
    "Testing": test_final["label"].value_counts()
})

split_summary = split_summary.rename(index=sentiment_labels)

display(split_summary)

# Verify that no reviews overlap between datasets
train_texts = set(train_final["clean_text"])
val_texts = set(val_final["clean_text"])
test_texts = set(test_final["clean_text"])

assert train_texts.isdisjoint(val_texts)
assert train_texts.isdisjoint(test_texts)
assert val_texts.isdisjoint(test_texts)

# Save dataset split summary
split_summary.to_csv("results/dataset_split_summary.csv")

print("\nAll dataset separation checks passed!")

### Dataset Split Findings

After text cleaning and duplicate removal, the cleaned training dataset was divided into training and validation sets using an 80/20 stratified split with a random seed of 42.

The final datasets contain:

- Training: 19,823 reviews
- Validation: 4,956 reviews
- Testing: 24,799 reviews
- Total: 49,578 reviews

All three datasets maintain approximately equal proportions of positive and negative reviews, indicating that preprocessing did not introduce substantial class imbalance.

Automated checks confirmed that no identical cleaned reviews overlap between the training, validation, and testing datasets.

The training set will be used to fit the model, the validation set will be used to evaluate modeling decisions, and the testing set will be reserved for final performance evaluation.

## 6. TF-IDF Feature Extraction

Machine learning models require numerical inputs rather than raw text. We use Term Frequency-Inverse Document Frequency (TF-IDF) to convert cleaned movie reviews into numerical features.

TF-IDF assigns greater importance to words that appear frequently in a particular review but are less common across the entire collection.

### Feature Extraction Decisions

- Use unigrams (individual words) and bigrams (two-word phrases).
- Limit the vocabulary to 30,000 features.
- Exclude terms that appear in fewer than five training reviews.
- Retain stop words to preserve sentiment-related expressions and negation.
- Fit the TF-IDF vectorizer only on the training dataset.
- Apply the fitted vectorizer to the validation and testing datasets.

Fitting the vectorizer only on training data prevents information from the validation and testing datasets from influencing the learned vocabulary and feature weights.

In [ ]:
# Initialize the TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=5,
    stop_words=None,
    sublinear_tf=True
)

# Fit TF-IDF on training data only
X_train = tfidf.fit_transform(train_final["clean_text"])

# Transform validation and testing data
X_val = tfidf.transform(val_final["clean_text"])
X_test = tfidf.transform(test_final["clean_text"])

# Extract sentiment labels
y_train = train_final["label"]
y_val = val_final["label"]
y_test = test_final["label"]

# Display feature matrix dimensions
print("Training feature matrix:", X_train.shape)
print("Validation feature matrix:", X_val.shape)
print("Testing feature matrix:", X_test.shape)

# Display vocabulary size
print("\nVocabulary size:", len(tfidf.get_feature_names_out()))

# Display sample vocabulary terms
print("\nSample vocabulary terms:")
print(tfidf.get_feature_names_out()[:20])

# Verify feature and label dimensions
assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)
assert X_test.shape[0] == len(y_test)

assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

print("\nAll feature extraction checks passed!")

### Feature Extraction Findings

The TF-IDF vectorizer successfully transformed the cleaned movie reviews into 30,000 numerical features using individual words and two-word phrases.

The resulting feature matrices contain:

- Training: 19,823 reviews and 30,000 features
- Validation: 4,956 reviews and 30,000 features
- Testing: 24,799 reviews and 30,000 features

All feature extraction checks passed, confirming that the feature matrices have consistent dimensions and align with their corresponding sentiment labels.

The vocabulary was learned exclusively from the training dataset to prevent data leakage. These numerical features will serve as inputs for sentiment classification.

## 7. Baseline Model

We use a majority-class classifier as our baseline model. This classifier always predicts the most common sentiment class in the training dataset, regardless of the review's content.

The baseline is implemented using Scikit-learn's DummyClassifier with the most_frequent strategy.

Because the dataset is nearly balanced, we expect the baseline to achieve approximately 50% accuracy.

The baseline provides a reference point for determining whether Logistic Regression meaningfully improves sentiment classification.

We evaluate the baseline on the validation dataset using accuracy, precision, recall, and F1-score. Final testing performance will be evaluated later alongside the Logistic Regression model.

In [ ]:
# Initialize the majority-class baseline model
baseline_model = DummyClassifier(
    strategy="most_frequent",
    random_state=RANDOM_STATE
)

# Train the baseline model
baseline_model.fit(X_train, y_train)

# Generate validation predictions
baseline_val_pred = baseline_model.predict(X_val)

# Calculate validation performance metrics
baseline_results = {
    "Model": "Majority-Class Baseline",
    "Accuracy": accuracy_score(y_val, baseline_val_pred),
    "Precision": precision_score(y_val, baseline_val_pred, zero_division=0),
    "Recall": recall_score(y_val, baseline_val_pred, zero_division=0),
    "F1-Score": f1_score(y_val, baseline_val_pred, zero_division=0)
}

# Display baseline performance
baseline_summary = pd.DataFrame([baseline_results])
display(baseline_summary.round(4))

# Display predicted sentiment distribution
print("\nPredicted sentiment distribution:")
print(pd.Series(baseline_val_pred).map(sentiment_labels).value_counts())

# Save baseline validation results
baseline_summary.to_csv(
    "results/baseline_validation_results.csv",
    index=False
)

### Baseline Model Findings

The majority-class baseline achieved the following validation results:

- Accuracy: 50.22%
- Precision: 50.22%
- Recall: 100.00%
- F1-Score: 66.86%

The baseline predicted all 4,956 validation reviews as positive because positive sentiment was the most frequent class in the training dataset.

Although the model achieved 100% recall for positive reviews, it failed to correctly identify any negative reviews. This demonstrates the limitations of a majority-class classifier and the importance of evaluating multiple performance metrics.

These results establish a reference point for evaluating whether Logistic Regression provides meaningful improvement in sentiment classification.

## 8. Logistic Regression Model

Logistic Regression is a supervised machine learning algorithm used for binary classification. In this project, it predicts whether a movie review expresses positive or negative sentiment based on its TF-IDF features.

### Model Configuration

- Algorithm: Logistic Regression
- Input: 30,000 TF-IDF features
- Output: Negative (0) or Positive (1)
- Regularization: L2
- Regularization parameter (C): 1.0
- Maximum iterations: 1,000
- Random seed: 42

Logistic Regression was selected because it is computationally efficient, interpretable, and effective for high-dimensional text classification tasks.

The model will be fitted exclusively on the training dataset and evaluated on the validation dataset before final testing. Its performance will be compared against the majority-class baseline using accuracy, precision, recall, and F1-score.

In [ ]:
# Initialize the Logistic Regression model
logistic_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Train the model using training data only
logistic_model.fit(X_train, y_train)

# Generate validation predictions
logistic_val_pred = logistic_model.predict(X_val)

# Calculate validation performance metrics
logistic_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val, logistic_val_pred),
    "Precision": precision_score(y_val, logistic_val_pred, zero_division=0),
    "Recall": recall_score(y_val, logistic_val_pred, zero_division=0),
    "F1-Score": f1_score(y_val, logistic_val_pred, zero_division=0)
}

# Display Logistic Regression performance
logistic_summary = pd.DataFrame([logistic_results])
display(logistic_summary.round(4))

# Compare Logistic Regression with the baseline
validation_comparison = pd.concat(
    [baseline_summary, logistic_summary],
    ignore_index=True
)

print("\nValidation Model Comparison:")
display(validation_comparison.round(4))

# Save validation comparison results
validation_comparison.to_csv(
    "results/validation_model_comparison.csv",
    index=False
)

# Verify prediction count and valid sentiment labels
assert len(logistic_val_pred) == len(y_val)
assert set(logistic_val_pred).issubset({0, 1})

print("\nAll model prediction checks passed!")

### Logistic Regression Findings

The Logistic Regression model achieved the following validation results:

- Accuracy: 90.17%
- Precision: 89.19%
- Recall: 91.52%
- F1-Score: 90.34%

Compared to the majority-class baseline, Logistic Regression improved accuracy by 39.95 percentage points and F1-score by 23.48 percentage points.

Although recall decreased from 100% to 91.52%, the Logistic Regression model provides a better balance between precision and recall and successfully identifies both positive and negative reviews.

These results demonstrate that combining TF-IDF feature extraction with Logistic Regression is effective for sentiment classification on the validation dataset.

All model prediction checks passed. Based on the strong validation performance, we retain the current model configuration for final testing.

## 9. Model Evaluation

The final Logistic Regression model and majority-class baseline are evaluated on the independent testing dataset of 24,799 movie reviews.

The testing dataset was not used to fit the TF-IDF vectorizer, train the models, or select the model configuration.

### Evaluation Metrics

- Accuracy: The proportion of all reviews correctly classified.
- Precision: The proportion of predicted positive reviews that are actually positive.
- Recall: The proportion of actual positive reviews correctly identified.
- F1-Score: The harmonic mean of precision and recall.
- Confusion Matrix: A breakdown of correct and incorrect predictions for both sentiment classes.

Positive sentiment (label 1) is treated as the positive class when calculating precision, recall, and F1-score.

We compare validation and testing performance to assess how well the model generalizes to unseen reviews. We also compare Logistic Regression against the majority-class baseline to measure improvement.

The testing dataset differs slightly from the original IMDb benchmark because duplicate and empty reviews were checked and duplicate reviews were removed during preprocessing.

In [ ]:
# Generate predictions on the independent testing dataset
baseline_test_pred = baseline_model.predict(X_test)
logistic_test_pred = logistic_model.predict(X_test)

# Define a function to calculate evaluation metrics
def evaluate_model(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }

# Evaluate both models on the testing dataset
test_comparison = pd.DataFrame([
    evaluate_model("Majority-Class Baseline", y_test, baseline_test_pred),
    evaluate_model("Logistic Regression", y_test, logistic_test_pred)
])

# Display final testing results
print("Final Testing Model Comparison:")
display(test_comparison.round(4))

# Compare Logistic Regression validation and testing performance
generalization_comparison = pd.DataFrame([
    evaluate_model("Validation", y_val, logistic_val_pred),
    evaluate_model("Testing", y_test, logistic_test_pred)
])

print("\nValidation vs. Testing Performance:")
display(generalization_comparison.round(4))

# Save final evaluation results
test_comparison.to_csv(
    "results/test_model_comparison.csv",
    index=False
)

generalization_comparison.to_csv(
    "results/generalization_comparison.csv",
    index=False
)

# Verify testing predictions
assert len(baseline_test_pred) == len(y_test)
assert len(logistic_test_pred) == len(y_test)
assert set(logistic_test_pred).issubset({0, 1})

print("\nAll final testing checks passed!")

### Final Testing Findings

The Logistic Regression model achieved the following results on the independent testing dataset of 24,799 movie reviews:

- Accuracy: 89.73%
- Precision: 89.20%
- Recall: 90.47%
- F1-Score: 89.83%

Compared to the majority-class baseline, Logistic Regression improved testing accuracy by 39.57 percentage points and F1-score by 23.02 percentage points.

Validation accuracy was 90.17%, while testing accuracy was 89.73%, representing a decrease of only 0.44 percentage points. Similarly, the F1-score decreased by only 0.51 percentage points.

The consistent validation and testing performance suggests that the model generalizes well to unseen IMDb movie reviews, with no substantial performance degradation.

These results demonstrate that combining TF-IDF with Logistic Regression provides an effective approach to binary movie review sentiment classification.

However, these results do not establish that the model will perform equally well on reviews from other platforms or domains.

All final testing checks passed successfully.

In [ ]:
# Calculate the confusion matrix
cm = confusion_matrix(y_test, logistic_test_pred, labels=[0, 1])

# Create a labeled confusion matrix
cm_df = pd.DataFrame(
    cm,
    index=["Negative", "Positive"],
    columns=["Negative", "Positive"]
)

# Create the confusion matrix visualization
plt.figure(figsize=(7, 5))

sns.heatmap(
    cm_df,
    annot=True,
    fmt=",d",
    cmap="Blues",
    cbar=False,
    annot_kws={"size": 14}
)

plt.title("Logistic Regression: Test Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.tight_layout()

# Save high-resolution visualization
plt.savefig(
    "figures/confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Display individual classification outcomes
tn, fp, fn, tp = cm.ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

# Save confusion matrix results
cm_df.to_csv("results/confusion_matrix.csv")

# Verify that all testing reviews are accounted for
assert cm.sum() == len(y_test)

print("\nConfusion matrix check passed!")

### Confusion Matrix Findings

The confusion matrix shows the following classification outcomes:

- True Negatives: 10,997
- False Positives: 1,363
- False Negatives: 1,185
- True Positives: 11,254

The model correctly classified 22,251 out of 24,799 testing reviews, achieving an overall accuracy of 89.73%.

The model correctly identified 90.47% of positive reviews and 88.97% of negative reviews, demonstrating strong performance across both sentiment classes.

The model produced slightly more false positives than false negatives, indicating a small tendency to classify reviews as positive.

A total of 2,548 reviews were misclassified. These incorrect predictions will be examined in the error analysis section to identify potential limitations and common classification challenges.

The confusion matrix verification confirmed that all testing reviews were accounted for.

In [ ]:
# Prepare testing metrics for visualization
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]

plot_data = test_comparison.set_index("Model")[metrics].T * 100

# Create model comparison bar chart
ax = plot_data.plot(
    kind="bar",
    figsize=(10, 6),
    color=["#9CA3AF", "#2563EB"],
    width=0.7
)

# Add percentage labels above each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3, fontsize=10)

# Customize chart
plt.title("Test Performance: Baseline vs. Logistic Regression")
plt.xlabel("Evaluation Metric")
plt.ylabel("Performance (%)")
plt.ylim(0, 115)
plt.xticks(rotation=0)
plt.legend(
    title="Model",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False
)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

# Save high-resolution visualization
plt.savefig(
    "figures/model_performance_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Model Comparison Findings

The model performance comparison demonstrates that Logistic Regression substantially outperforms the majority-class baseline in accuracy, precision, and F1-score.

Logistic Regression achieved 89.73% testing accuracy compared to the baseline's 50.16%, representing an improvement of 39.57 percentage points.

Although the baseline achieved 100% recall by predicting every review as positive, Logistic Regression provides a better balance between precision and recall while successfully identifying both sentiment classes.

These results support the effectiveness of combining TF-IDF feature extraction with Logistic Regression for movie review sentiment classification.

## 10. Error Analysis

Although Logistic Regression achieved strong overall performance, it incorrectly classified 2,548 testing reviews.

Error analysis is conducted to better understand the model's limitations and identify common patterns among misclassified reviews.

### Error Analysis Approach

- Identify incorrectly classified reviews.
- Separate false positives and false negatives.
- Examine prediction probabilities.
- Review examples of incorrect predictions.
- Identify potential reasons for classification errors.

Potential classification challenges include sarcasm, negation, mixed sentiment, ambiguous language, and context-dependent expressions.

Because TF-IDF represents text primarily through word and phrase frequencies, it may struggle to capture deeper contextual meaning and complex language patterns.

The purpose of this analysis is to understand model weaknesses rather than modify the model based on the testing dataset.

In [ ]:
# Create a dataframe containing testing predictions
error_analysis = test_final[["clean_text", "label"]].copy()

# Add predicted labels and positive sentiment probabilities
error_analysis["predicted_label"] = logistic_test_pred
error_analysis["positive_probability"] = logistic_model.predict_proba(X_test)[:, 1]

# Identify incorrectly classified reviews
misclassified = error_analysis[
    error_analysis["label"] != error_analysis["predicted_label"]
].copy()

# Categorize classification errors
misclassified["error_type"] = np.where(
    misclassified["label"] == 0,
    "False Positive",
    "False Negative"
)

# Calculate model confidence
misclassified["confidence"] = np.maximum(
    misclassified["positive_probability"],
    1 - misclassified["positive_probability"]
)

# Display error counts
print("Total misclassified reviews:", len(misclassified))

print("\nError type distribution:")
print(misclassified["error_type"].value_counts())

# Display confidence summary by error type
print("\nConfidence Summary:")
display(
    misclassified.groupby("error_type")["confidence"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

# Save error results without raw review text
misclassified.drop(columns=["clean_text"]).to_csv(
    "results/misclassified_reviews.csv",
    index_label="test_review_index"
)

# Verify error counts against the confusion matrix
assert len(misclassified) == fp + fn

print("\nError analysis check passed!")

### Error Distribution Findings

The Logistic Regression model incorrectly classified 2,548 testing reviews, representing 10.27% of the testing dataset.

Of these errors:

- 1,363 (53.49%) were false positives.
- 1,185 (46.51%) were false negatives.

The average prediction confidence was 62.98% for false positives and 62.09% for false negatives.

The median confidence scores were approximately 60% for both error types, suggesting that many incorrect predictions occurred near the model's classification threshold.

However, the maximum confidence reached 97.01% for false positives and 94.03% for false negatives. This demonstrates that the model can make incorrect predictions even when assigning high probabilities to its predicted class.

These findings highlight the importance of interpreting prediction probabilities cautiously rather than treating them as guarantees of correctness.

In [ ]:
# Select three high-confidence errors from each error type
false_positive_examples = misclassified[
    misclassified["error_type"] == "False Positive"
].nlargest(3, "confidence")

false_negative_examples = misclassified[
    misclassified["error_type"] == "False Negative"
].nlargest(3, "confidence")

# Combine selected examples
error_examples = pd.concat([
    false_positive_examples,
    false_negative_examples
])

# Display misclassified review examples
for index, row in error_examples.iterrows():
    print("=" * 80)
    print("Review Index:", index)
    print("Error Type:", row["error_type"])
    print("Actual Sentiment:", sentiment_labels[row["label"]])
    print("Predicted Sentiment:", sentiment_labels[row["predicted_label"]])
    print(f"Prediction Confidence: {row['confidence']:.2%}")
    print("\nReview:")
    print(row["clean_text"])
    print()

# Save example metadata without review text
error_examples.drop(columns=["clean_text"]).to_csv(
    "results/error_examples.csv",
    index_label="test_review_index"
)

### Misclassified Review Findings

Six high-confidence misclassified reviews were examined, including three false positives and three false negatives.

The following patterns were identified:

- Mixed Sentiment: Some reviews contained both positive and negative opinions, making overall sentiment difficult to classify.
- Sentiment Shifts: Some reviews began with negative criticism but concluded with a positive overall impression.
- Contextual Limitations: TF-IDF emphasizes word and phrase frequencies but does not fully capture how opinions develop throughout a review.
- Potential Label Inconsistencies: Several reviews expressed sentiments that appeared inconsistent with their assigned dataset labels.
- Rating and Text Disagreement: One review contained substantial criticism despite providing a numerical rating of 7/10.

These examples suggest that classification errors may result from both model limitations and ambiguity within the dataset.

The analysis also demonstrates that high prediction confidence does not guarantee correctness.

Because only six high-confidence errors were examined, these findings should not be generalized to all misclassified reviews. Future improvements could include examining a larger random sample of errors and exploring models that better capture contextual meaning.

In [ ]:
# Test the sentiment classification model on new reviews
sample_reviews = [
    "This movie was absolutely amazing. I loved every minute!",
    "This was a terrible movie and a complete waste of time.",
    "The acting was good, but the storyline was disappointing.",
    "I did not enjoy this movie."
]

# Clean and transform sample reviews
cleaned_samples = [clean_text(review) for review in sample_reviews]
sample_features = tfidf.transform(cleaned_samples)

# Generate predictions
sample_predictions = logistic_model.predict(sample_features)
sample_probabilities = logistic_model.predict_proba(sample_features)

# Display results
sample_results = pd.DataFrame({
    "Review": sample_reviews,
    "Predicted Sentiment": [
        sentiment_labels[label] for label in sample_predictions
    ],
    "Confidence": sample_probabilities.max(axis=1).round(4)
})

display(sample_results)

# Test basic cleaning edge cases
assert clean_text("") == ""
assert clean_text(None) == ""
assert clean_text("   ") == ""
assert clean_text("<br>Great Movie!") == "great movie!"

# Verify prediction functionality
assert len(sample_predictions) == len(sample_reviews)
assert set(sample_predictions).issubset({0, 1})

print("\nAll software functionality checks passed!")

### Software Testing Findings

The model successfully classified four new sample reviews containing positive sentiment, negative sentiment, mixed opinions, and negation.

Additional checks confirmed that the text cleaning function correctly handled empty inputs, missing values, whitespace, and HTML tags.

All software functionality checks passed. These tests demonstrate that the model can process new text inputs, although they do not establish performance outside the IMDb dataset.

## 11. Limitations and Responsible Use

### Model Limitations

- **Limited Contextual Understanding:** TF-IDF captures word and phrase frequencies but does not fully understand context, tone, or complex language.
- **Mixed Sentiment:** The model may struggle with reviews containing both positive and negative opinions.
- **Dataset Quality:** Some reviews appear to express sentiments that differ from their assigned labels.
- **Limited Generalizability:** The model was trained on IMDb movie reviews and may not perform equally well on other platforms or types of text.
- **Binary Classification:** The model only predicts positive or negative sentiment and cannot classify neutral sentiment.
- **Prediction Confidence:** High prediction confidence does not guarantee an accurate classification.

### Responsible Use

The model is intended for educational purposes and general movie review sentiment analysis. Its predictions should not be treated as definitive interpretations of individual opinions.

Human review is recommended when accurate sentiment interpretation is important. Prediction probabilities should also be interpreted cautiously.

To support responsible data handling, the original dataset is accessed through Hugging Face rather than redistributed through the GitHub repository.

### Future Improvements

- Explore contextual language models such as BERT.
- Examine a larger and more representative sample of classification errors.
- Evaluate performance on reviews from other platforms.
- Investigate potential dataset labeling inconsistencies.
- Explore probability calibration and neutral sentiment classification.

## 12. Conclusion

This project investigated the effectiveness of combining TF-IDF feature extraction with Logistic Regression to classify IMDb movie reviews as positive or negative.

After data cleaning, duplicate removal, and dataset separation, a total of 49,578 reviews were retained. The cleaned reviews were transformed into 30,000 numerical features using TF-IDF.

### Key Results

- Baseline Testing Accuracy: 50.16%
- Logistic Regression Validation Accuracy: 90.17%
- Logistic Regression Testing Accuracy: 89.73%
- Logistic Regression Testing Precision: 89.20%
- Logistic Regression Testing Recall: 90.47%
- Logistic Regression Testing F1-Score: 89.83%

Logistic Regression improved testing accuracy by 39.57 percentage points compared to the majority-class baseline. The small difference between validation and testing performance suggests that the model generalizes well to unseen IMDb reviews.

Error analysis identified mixed sentiment, limited contextual understanding, and potential dataset labeling inconsistencies as important classification challenges.

Overall, this project demonstrates that traditional machine learning techniques can provide an effective, interpretable, and computationally efficient approach to movie review sentiment classification. Future research could explore contextual language models and additional datasets to improve classification performance and generalizability.

## 13. AI-Use Statement

We used ChatGPT to assist with project organization, code development, debugging, and interpretation of results. We reviewed and adapted its suggestions, tested the code in Google Colab, and evaluated the model using actual outputs. We take responsibility for the final analysis and findings.